### 1. Data & LLM Classification

#### 1.1. Imports and Project Paths

In [46]:
import json
import os
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"

TAXONOMY_PATH = DATA_DIR / "taxonomy.json"
COMPLAINTS_PATH = DATA_DIR / "complaints_sample.jsonl"

#### 1.2. Load Taxonomy and Complaints

In [47]:
# Taxonomy 

with open(TAXONOMY_PATH, "r", encoding="utf-8") as file:
    taxonomy_data = json.load(file)

taxonomy = taxonomy_data["taxonomy"]

# Complaints 

complaints_df = pd.read_json(COMPLAINTS_PATH, lines=True)

complaints_df["complaint_id"] = (
    complaints_df["complaint_id"]
    .astype(str)
    .str.zfill(3))


#### 1.3. Prepare taxonomy for the prompt

In [33]:
# Transform our Python/JSON structure into a text version that we can send to the LLM.
taxonomy_text = json.dumps(
    taxonomy,
    indent=2,
    ensure_ascii=False)

#### 1.4. Build the Prompt for the LLM

In [34]:
def build_classification_prompt(complaint):
    
    prompt = f"""
You are a bank complaint classification assistant.

Your task is to classify a customer complaint according to the
official banking complaint taxonomy provided below.

The taxonomy follows this hierarchy:

Product
    -> Matter

Rules:

1. Analyse the complete complaint and its context.
2. Identify the main issue described by the customer.
3. Select the Product that best represents the main issue.
4. Select a Matter belonging to that Product.
5. Use ONLY Products and Matters from the official taxonomy.
6. Never invent, modify, or combine Products or Matters.
7. Return a confidence value between 0.0 and 1.0.
8. The confidence represents your confidence in the classification.

Official taxonomy:

{taxonomy_text}

Complaint:

Complaint ID:
{complaint["complaint_id"]}

Complaint text:
{complaint["complaint_text"]}

Return ONLY a valid JSON object with exactly these fields:

{{
    "complaint_id": "string",
    "product": "string",
    "matter": "string",
    "confidence": 0.0
}}

Do not include explanations or Markdown.
"""

    return prompt

#### 1.5. Open Router (LLM) Configuration 

In [35]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY is not set."
    )

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

MODEL = "nvidia/nemotron-3-super-120b-a12b:free"

#### 1.6. LLM function

In [36]:
# This function defines hoe how python talks with the LLM

def call_llm(prompt):
    
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0
    }

    response = requests.post(
        OPENROUTER_URL,
        headers=headers,
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    result = response.json()

    return result["choices"][0]["message"]["content"]

### 2. Business Logic & Evaluation

#### 2.1. Apply the business rule

In [60]:
# Define the confidence threshold

CONFIDENCE_THRESHOLD = 0.90


# Apply the business rule

if classification["confidence"] >= CONFIDENCE_THRESHOLD:
    classification["routing_status"] = "Approved"
else:
    classification["routing_status"] = "Needs Review"

classification

{'complaint_id': '050',
 'product': 'Guarantees',
 'matter': 'Enforcement / calling of the guarantee',
 'confidence': 0.95,
 'routing_status': 'Approved'}

#### 2.2. Select a sample for the experiment

In [55]:
# Select a small sample due to the API request limit

SAMPLE_SIZE = 50

complaints_sample = complaints_df.head(SAMPLE_SIZE).copy()

#### 2.3. Classify the sample

In [56]:
results = []

for _, complaint in complaints_sample.iterrows():
    
    prompt = build_classification_prompt(complaint)
    
    raw_response = call_llm(prompt)
    
    classification = json.loads(raw_response)
    
    results.append(classification)

predictions_df = pd.DataFrame(results)

predictions_df

,complaint_id,product,matter,confidence
0,001,Direct debits,SEPA direct debits,0.95
1,002,Guarantees,Cancellation,0.95
2,003,Current / savings accounts,Change of account holder,0.95
3,004,Term deposits,Interest calculation and settlement,0.95
4,005,Electronic money,"Prepaid funds: loading, reloading and redemption",0.95
5,006,Payment instruments,Cheques,0.95
6,007,Garnishments and attachments,Public-authority debt attachments,0.95
7,008,Other complaints,Delay / lack of attention / deficient service,0.95
8,009,Mortgage loans,Fees,0.95
9,010,Unsecured loans to SMEs / self-employed persons,COVID-19 ICO guarantees - early repayment,0.95


#### 2.4. LLM Response Validation

In [58]:
for _, prediction in predictions_df.iterrows():

    # Check required fields

    required_fields = {
        "complaint_id",
        "product",
        "matter",
        "confidence"
    }

    missing_fields = required_fields - set(prediction.index)

    if missing_fields:
        raise ValueError(
            f"Missing fields: {missing_fields}"
        )


    # Check confidence

    confidence = prediction["confidence"]

    if not isinstance(confidence, (int, float)):
        raise ValueError(
            "Confidence must be numeric."
        )

    if not 0 <= confidence <= 1:
        raise ValueError(
            "Confidence must be between 0 and 1."
        )


    # Check product

    if prediction["product"] not in valid_products:
        raise ValueError(
            f"Invalid product: {prediction['product']}"
        )


    # Check matter

    product = prediction["product"]
    matter = prediction["matter"]

    if matter not in valid_matters_by_product[product]:
        raise ValueError(
            f"Matter '{matter}' does not belong to "
            f"product '{product}'."
        )


print("All predictions passed validation.")

All predictions passed validation.


#### 2.2. Apply the Business Rule to All Predictions

In [61]:
predictions_df["routing_status"] = predictions_df["confidence"].apply(
    lambda confidence:
        "Approved"
        if confidence >= CONFIDENCE_THRESHOLD
        else "Needs Review"
)

predictions_df

,complaint_id,product,matter,confidence,routing_status
0,001,Direct debits,SEPA direct debits,0.95,Approved
1,002,Guarantees,Cancellation,0.95,Approved
2,003,Current / savings accounts,Change of account holder,0.95,Approved
3,004,Term deposits,Interest calculation and settlement,0.95,Approved
4,005,Electronic money,"Prepaid funds: loading, reloading and redemption",0.95,Approved
5,006,Payment instruments,Cheques,0.95,Approved
6,007,Garnishments and attachments,Public-authority debt attachments,0.95,Approved
7,008,Other complaints,Delay / lack of attention / deficient service,0.95,Approved
8,009,Mortgage loans,Fees,0.95,Approved
9,010,Unsecured loans to SMEs / self-employed persons,COVID-19 ICO guarantees - early repayment,0.95,Approved


### 3. Evaluation

##### 3.1. Load Human Evaluation Dataset

In [62]:
# Load the human evaluation (ground truth).
# This file contains the classification that we consider correct for each complaint. It is kept separate from the LLM predictions
# so that the model is evaluated against an independent reference.

GROUND_TRUTH_PATH = DATA_DIR / "ground_truth.jsonl"

ground_truth_df = pd.read_json(
    GROUND_TRUTH_PATH,
    lines=True
)

# Keep complaint IDs as strings with the same format used in the predictions, so that the two datasets can be matched correctly.

ground_truth_df["complaint_id"] = (
    ground_truth_df["complaint_id"]
    .astype(str)
    .str.zfill(3)
)

##### 3.2. LLM Classification Vs Human Classification

In [63]:
# Compare the LLM predictions with the human ground truth.
#
# The human evaluation is our reference (ground truth).
# We compare the model at two levels:
#   1. Product: did the LLM identify the correct banking product?
#   2. Matter: did it identify the correct issue within that product?
#
# A classification is considered fully correct only when BOTH
# Product and Matter match the human evaluation.

evaluation_df = predictions_df.merge(
    ground_truth_df,
    on="complaint_id",
    how="inner"
)


# Check Product predictions

evaluation_df["product_correct"] = (
    evaluation_df["product"]
    == evaluation_df["expected_product"]
)


# Check Matter predictions

evaluation_df["matter_correct"] = (
    evaluation_df["matter"]
    == evaluation_df["expected_matter"]
)


# Check the complete classification

evaluation_df["classification_correct"] = (
    evaluation_df["product_correct"]
    & evaluation_df["matter_correct"]
)


# Calculate the evaluation metrics.
#
# mean() works because True = 1 and False = 0.
# Therefore, the average of these columns gives us the accuracy.

product_accuracy = evaluation_df["product_correct"].mean()

matter_accuracy = evaluation_df["matter_correct"].mean()

overall_accuracy = (
    evaluation_df["classification_correct"].mean()
)


# Count correct and incorrect predictions

product_correct_count = (
    evaluation_df["product_correct"].sum()
)

matter_correct_count = (
    evaluation_df["matter_correct"].sum()
)

classification_correct_count = (
    evaluation_df["classification_correct"].sum()
)

total_complaints = len(evaluation_df)

product_incorrect_count = (
    total_complaints - product_correct_count
)

matter_incorrect_count = (
    total_complaints - matter_correct_count
)

classification_incorrect_count = (
    total_complaints - classification_correct_count
)


# Display the main results

print("Evaluation Results")
print("------------------")

print(
    f"Product Accuracy: "
    f"{product_accuracy:.2%} "
    f"({product_correct_count}/{total_complaints})"
)

print(
    f"Matter Accuracy: "
    f"{matter_accuracy:.2%} "
    f"({matter_correct_count}/{total_complaints})"
)

print(
    f"Overall Accuracy: "
    f"{overall_accuracy:.2%} "
    f"({classification_correct_count}/{total_complaints})"
)

print()

print("Errors")
print("------")

print(
    f"Incorrect Products: "
    f"{product_incorrect_count}"
)

print(
    f"Incorrect Matters: "
    f"{matter_incorrect_count}"
)

print(
    f"Incorrect Classifications: "
    f"{classification_incorrect_count}"
)


# Show the detailed comparison.
#
# This table is useful for error analysis because it lets us see
# exactly what the human expected and what the LLM predicted.

evaluation_df[
    [
        "complaint_id",
        "expected_product",
        "product",
        "product_correct",
        "expected_matter",
        "matter",
        "matter_correct",
        "confidence",
        "classification_correct"
    ]
]

Evaluation Results
------------------
Product Accuracy: 94.00% (47/50)
Matter Accuracy: 94.00% (47/50)
Overall Accuracy: 94.00% (47/50)

Errors
------
Incorrect Products: 3
Incorrect Matters: 3
Incorrect Classifications: 3


,complaint_id,expected_product,product,product_correct,expected_matter,matter,matter_correct,confidence,classification_correct
0,001,Direct debits,Direct debits,True,SEPA direct debits,SEPA direct debits,True,0.95,True
1,002,Guarantees,Guarantees,True,Cancellation,Cancellation,True,0.95,True
2,003,Current / savings accounts,Current / savings accounts,True,Change of account holder,Change of account holder,True,0.95,True
3,004,Term deposits,Term deposits,True,Interest calculation and settlement,Interest calculation and settlement,True,0.95,True
4,005,Electronic money,Electronic money,True,"Prepaid funds: loading, reloading and redemption","Prepaid funds: loading, reloading and redemption",True,0.95,True
5,006,Payment instruments,Payment instruments,True,Cheques,Cheques,True,0.95,True
6,007,Garnishments and attachments,Garnishments and attachments,True,Public-authority debt attachments,Public-authority debt attachments,True,0.95,True
7,008,Other complaints,Other complaints,True,Delay / lack of attention / deficient service,Delay / lack of attention / deficient service,True,0.95,True
8,009,Mortgage loans,Mortgage loans,True,Fees,Fees,True,0.95,True
9,010,Unsecured loans to SMEs / self-employed persons,Unsecured loans to SMEs / self-employed persons,True,COVID-19 ICO guarantees - early repayment,COVID-19 ICO guarantees - early repayment,True,0.95,True


## Export evaluation results

The evaluation results are exported to a JSON file so they can be used
outside the notebook, particularly by the HTML dashboard.

The JSON contains both the overall evaluation metrics and the detailed
results for each complaint. For each complaint, we keep the original
complaint text, the human ground truth, the LLM prediction, the model
confidence, the evaluation results, and the routing status.

Keeping this information together allows the dashboard to:
- present the overall model performance;
- show complaints classified as `Needs Review`;
- identify classification errors;
- compare the LLM prediction with the human ground truth;
- analyse the model confidence for each complaint.

The notebook is therefore responsible for the LLM experiment and its
evaluation, while the dashboard will use the exported JSON as its data
source.

In [64]:
# Add the original complaint text
evaluation_df = evaluation_df.merge(
    complaints_df[
        ["complaint_id", "complaint_text"]
    ],
    on="complaint_id",
    how="left"
)


# Apply the routing rule
evaluation_df["routing_status"] = evaluation_df["confidence"].apply(
    lambda confidence:
        "Approved"
        if confidence >= CONFIDENCE_THRESHOLD
        else "Needs Review"
)


# Count routing results
approved_count = (
    evaluation_df["routing_status"] == "Approved"
).sum()

needs_review_count = (
    evaluation_df["routing_status"] == "Needs Review"
).sum()


# Build the final results structure
results = {
    "evaluation": {
        "total_complaints": total_complaints,
        "product_accuracy": product_accuracy,
        "matter_accuracy": matter_accuracy,
        "overall_accuracy": overall_accuracy,
        "product_correct": int(product_correct_count),
        "product_incorrect": int(product_incorrect_count),
        "matter_correct": int(matter_correct_count),
        "matter_incorrect": int(matter_incorrect_count),
        "classification_correct": int(
            classification_correct_count
        ),
        "classification_incorrect": int(
            classification_incorrect_count
        )
    },
    
    "routing": {
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "approved_count": int(approved_count),
        "needs_review_count": int(needs_review_count)
    },
    
    "complaints": []
}


# Add individual complaint results
for _, row in evaluation_df.iterrows():
    
    results["complaints"].append({
        "complaint_id": row["complaint_id"],
        "complaint_text": row["complaint_text"],
        "expected": {
            "product": row["expected_product"],
            "matter": row["expected_matter"]
        },
        "predicted": {
            "product": row["product"],
            "matter": row["matter"],
            "confidence": row["confidence"]
        },
        "evaluation": {
            "product_correct": bool(row["product_correct"]),
            "matter_correct": bool(row["matter_correct"]),
            "classification_correct": bool(
                row["classification_correct"]
            )
        },
        "routing_status": row["routing_status"]
    })


# Export results
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

RESULTS_PATH = RESULTS_DIR / "evaluation_results.json"

with open(RESULTS_PATH, "w", encoding="utf-8") as file:
    json.dump(
        results,
        file,
        indent=2,
        ensure_ascii=False
    )

print(f"Results exported successfully to: {RESULTS_PATH}")

Results exported successfully to: ..\results\evaluation_results.json
